In [ ]:
import numpy as np
import pandas as pd
from protossl.defines import PTBXL_TARGETS, CINC_TARGETS, CODE15_TARGETS, HEEDB_TARGETS

from pathlib import Path

run_dir = Path("/opt/gpu_working/steven/protossl-ecg-outputs")
ptbxl_dir = run_dir / "experiments-seed42/runs-ptbxl"
cinc_dir = run_dir / "experiments-seed42/runs-cinc"
code15_dir = run_dir / "experiments-seed42/runs-code15"

PPL = 14

In [ ]:
(
    list(HEEDB_TARGETS).index("RIGHT BUNDLE BRANCH BLOCK"),
    PTBXL_TARGETS.index("CRBBB"),
    CINC_TARGETS.index("CRBBB"),
    CODE15_TARGETS.index("RBBB"),
)

In [ ]:
heedb_proj = pd.read_csv(run_dir / "protossl-heedb/project-prototypes/latest/projection_metadata.csv")

ptbxl_assign = pd.read_csv(ptbxl_dir / "protossl-heedb-pila/learn-prototype-assignments/latest/assignment_metadata.csv")
ptbxl_proj = pd.read_csv(ptbxl_dir / "protossl-heedb-pila/project-prototypes-supervised/latest/projection_metadata.csv")
ptbxl_ft_proj = pd.read_csv(ptbxl_dir / "protossl-heedb-pila-ft/project-prototypes-supervised/latest/projection_metadata.csv")

cinc_assign = pd.read_csv(cinc_dir / "protossl-heedb-pila/learn-prototype-assignments/latest/assignment_metadata.csv")
cinc_proj = pd.read_csv(cinc_dir / "protossl-heedb-pila/project-prototypes-supervised/latest/projection_metadata.csv")
cinc_ft_proj = pd.read_csv(cinc_dir / "protossl-heedb-pila-ft/project-prototypes-supervised/latest/projection_metadata.csv")

code15_assign = pd.read_csv(code15_dir / "protossl-heedb-pila/learn-prototype-assignments/latest/assignment_metadata.csv")
code15_proj = pd.read_csv(code15_dir / "protossl-heedb-pila/project-prototypes-supervised/latest/projection_metadata.csv")
code15_ft_proj = pd.read_csv(code15_dir / "protossl-heedb-pila-ft/project-prototypes-supervised/latest/projection_metadata.csv")

In [ ]:
ptbxl_protos = set(ptbxl_assign.loc[ptbxl_assign["label_name"] == "CRBBB", "prototype_idx"])
cinc_protos = set(cinc_assign.loc[cinc_assign["label_name"] == "CRBBB", "prototype_idx"])
code15_protos = set(code15_assign.loc[code15_assign["label_name"] == "RBBB", "prototype_idx"])

In [ ]:
sets = {
    "PTB-XL": ptbxl_protos,
    "CinC": cinc_protos,
    "CODE-15%": code15_protos,
}
names = list(sets)
universe = sorted(set().union(*sets.values()))

flags = pd.DataFrame(
    {name: [x in s for x in universe] for name, s in sets.items()},
    index=pd.Index(universe, name="value"),
)

In [ ]:
regions = (
    flags.groupby(names)
    .agg(count=("PTB-XL", "size"), members=(flags.columns[0], lambda s: list(s.index)))
    .reset_index()
)
regions["region"] = ""
for i in range(len(regions)):
    temp = [n for n in names if regions.loc[i, n]]
    regions.loc[i, "region"] = " & ".join(temp) + (" only" if len(temp) == 1 else "")
regions["num_regions"] = regions[names].sum(axis=1)

out = regions.sort_values(["num_regions", "region"]).reset_index(drop=True)
n = len(out)
assert out["count"].sum() == len(ptbxl_protos | cinc_protos | code15_protos)
out.loc[n, "count"] = out["count"].sum()
out.loc[n, "region"] = "Total"
out["count"] = out["count"].astype(int)
out[["region", "count", "members"]]

In [ ]:
out[["region", "count"]].rename(columns={"region": "Datasets", "count": "Count"})

In [ ]:
# !pip install matplotlib-venn
from matplotlib_venn import venn3

venn3(
    tuple(sets.values()),
    set_labels=tuple(sets.keys()),
)

In [ ]:
proto_idx = 394

In [ ]:
heedb_proj_sample = heedb_proj.loc[proto_idx]

In [ ]:
ptbxl_slot = ptbxl_assign[ptbxl_assign["prototype_idx"] == proto_idx]
assert len(ptbxl_slot) == 1
ptbxl_proj_sample = ptbxl_proj.loc[ptbxl_slot.iloc[0].name]
ptbxl_ft_proj_sample = ptbxl_ft_proj.loc[ptbxl_slot.iloc[0].name]

In [ ]:
cinc_slot = cinc_assign[cinc_assign["prototype_idx"] == proto_idx]
assert len(cinc_slot) == 1
cinc_proj_sample = cinc_proj.loc[cinc_slot.iloc[0].name]
cinc_ft_proj_sample = cinc_ft_proj.loc[cinc_slot.iloc[0].name]

In [ ]:
code15_slot = code15_assign[code15_assign["prototype_idx"] == proto_idx]
assert len(code15_slot) == 1
code15_proj_sample = code15_proj.loc[code15_slot.iloc[0].name]
code15_ft_proj_sample = code15_ft_proj.loc[code15_slot.iloc[0].name]

In [ ]:
from protossl.datasets import HeedbECGDataset, PtbxlECGDataset, CincECGDataset, Code15ECGDataset
from protossl.plotting import plot_ecg

heedb_ds = HeedbECGDataset(dataset_path="/opt/gpudata/ecg/heedb", split="train", sampling_rate=100)
ptbxl_ds = PtbxlECGDataset(dataset_path="/opt/gpudata/ecg/ptb-xl", split="train", sampling_rate=100)
cinc_ds = CincECGDataset(dataset_path="/opt/gpudata/ecg/cinc-2020", split="train", sampling_rate=100)
code15_ds = Code15ECGDataset(dataset_path="/opt/gpudata/ecg/code15", split="train", sampling_rate=100)

In [ ]:
heedb_fig = plot_ecg(dataset=heedb_ds, sample_id=int(heedb_proj_sample["ecg_id"]), chunk_idx=int(heedb_proj_sample["chunk_idx"]))
heedb_fig.savefig("figs/heedb-rbbb.png")

In [ ]:
ptbxl_fig = plot_ecg(dataset=ptbxl_ds, sample_id=int(ptbxl_proj_sample["ecg_id"]), chunk_idx=int(ptbxl_proj_sample["chunk_idx"]))
ptbxl_fig.savefig("figs/ptbxl-rbbb.png")

In [ ]:
ptbxl_ft_fig = plot_ecg(dataset=ptbxl_ds, sample_id=int(ptbxl_ft_proj_sample["ecg_id"]), chunk_idx=int(ptbxl_ft_proj_sample["chunk_idx"]))
ptbxl_ft_fig.savefig("figs/ptbxl-ft-rbbb.png")

In [ ]:
cinc_fig = plot_ecg(dataset=cinc_ds, sample_id=int(cinc_proj_sample["ecg_id"]), chunk_idx=int(cinc_proj_sample["chunk_idx"]))
cinc_fig.savefig("figs/cinc-rbbb.png")

In [ ]:
cinc_ft_fig = plot_ecg(dataset=cinc_ds, sample_id=int(cinc_ft_proj_sample["ecg_id"]), chunk_idx=int(cinc_ft_proj_sample["chunk_idx"]))
cinc_ft_fig.savefig("figs/cinc-ft-rbbb.png")

In [ ]:
code15_fig = plot_ecg(dataset=code15_ds, sample_id=int(code15_proj_sample["ecg_id"]), chunk_idx=int(code15_proj_sample["chunk_idx"]))
code15_fig.savefig("figs/code15-rbbb.png")

In [ ]:
code15_ft_fig = plot_ecg(dataset=code15_ds, sample_id=int(code15_ft_proj_sample["ecg_id"]), chunk_idx=int(code15_ft_proj_sample["chunk_idx"]))
code15_ft_fig.savefig("figs/code15-ft-rbbb.png")

In [ ]:
import torch

In [ ]:
sd = torch.load("../scripts/ecgfounder-checkpoint/12_lead_ECGFounder.pth", map_location="cpu", weights_only=False)["state_dict"]

In [ ]:
sd["dense.weight"].shape

In [ ]:
sd["dense.bias"].shape